In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q langchain-text-splitters

In [1]:
import os
import re
import torch
import gc
import pandas as pd
import numpy as np
import json
import pickle
from tqdm import tqdm
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# ====================  TEXT EXTRACTION ====================
def extract_text_from_htm(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')
        raw_text = soup.get_text(separator='\n')

        lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
        text = '\n'.join(lines)
        clean_text = re.sub(r'\n\s*\n+', '\n\n', text).strip()
        return clean_text
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# Generate Embeddings

In [ ]:
import time

books_folder = "/content/drive/MyDrive/100_books/"
save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa5_nano"

# model_name = "jinaai/jina-embeddings-v5-text-nano"
model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"
model = SentenceTransformer(model_name,
                             trust_remote_code=True,
                             device='cuda' if torch.cuda.is_available() else 'cpu'
                             )

# Fixed chunk settings for fair comparison
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=90,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
actions = []

# --- Tracking variables ---
total_chunks_created = 0
total_embedding_time = 0.0  # time spent only on model.encode()

for file_name in tqdm(htm_files):          # Start with 20 books
    base_name = file_name.replace(".htm", "").replace(".html", "")
    embedding_file = os.path.join(save_folder, f"{base_name}_embeddings.pkl")

    # Skip if already processed
    if os.path.exists(embedding_file):
        print(f"✅ Already processed: {file_name}")
        continue

    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    # if len(text) < 300:
    #     continue

    # chunks = chunk_text(text)
    chunks = text_splitter.split_text(text)
    total_chunks_created += len(chunks)

    # --- Time only the embedding generation step ---
    start_time = time.time()
    embeddings = model.encode(chunks, batch_size=64, show_progress_bar=False)
    elapsed = time.time() - start_time
    total_embedding_time += elapsed

    print(f"⏱️ {file_name}: {len(chunks)} chunks embedded in {elapsed:.2f}s")

    # Save to Drive
    data_to_save = {
        "chunks": chunks,
        "embeddings": embeddings,
        "file_name": file_name,
        "book_title": base_name
    }

    with open(embedding_file, "wb") as f:
        pickle.dump(data_to_save, f)

    print(f"💾 Saved embeddings for {file_name}")

    # Clean memory
    del chunks, embeddings
    torch.cuda.empty_cache()
    gc.collect()

# --- Final summary ---
print("\n" + "="*50)
print(f"📊 Total chunks created: {total_chunks_created}")
print(f"⏱️ Total embedding generation time: {total_embedding_time:.2f} seconds "
      f"({total_embedding_time/60:.2f} minutes)")
if total_chunks_created > 0:
    print(f"⚡ Average time per chunk: {total_embedding_time/total_chunks_created:.4f} seconds")
print("="*50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.22k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

...

⏱️ کشکول فاطمی -()_19793.html: 555 chunks embedded in 16.29s
💾 Saved embeddings for کشکول فاطمی -()_19793.html


 99%|█████████▉| 99/100 [25:55<00:14, 14.32s/it]

⏱️ گلشن ياس نبي -()_16106.html: 1116 chunks embedded in 33.26s
💾 Saved embeddings for گلشن ياس نبي -()_16106.html


100%|██████████| 100/100 [26:29<00:00, 15.90s/it]


📊 Total chunks created: 54381
⏱️ Total embedding generation time: 1514.42 seconds (25.24 minutes)
⚡ Average time per chunk: 0.0278 seconds


# Generate and save embeddings with jinaa3

In [3]:
!pip install -q \
  sentence-transformers==3.3.1 \
  transformers==4.47.1 \
  tokenizers==0.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.4 MB/s eta 0:00:00


In [ ]:
import time

books_folder = "/content/drive/MyDrive/100_books/"
save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa3"

model_name = "jinaai/jina-embeddings-v3"

model = SentenceTransformer(model_name,
                             trust_remote_code=True,
                             device='cuda' if torch.cuda.is_available() else 'cpu'
                             )

# Fixed chunk settings for fair comparison
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=90,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
actions = []

# --- Tracking variables ---
total_chunks_created = 0
total_embedding_time = 0.0  # time spent only on model.encode()

for file_name in tqdm(htm_files):          # Start with 20 books
    base_name = file_name.replace(".htm", "").replace(".html", "")
    embedding_file = os.path.join(save_folder, f"{base_name}_embeddings.pkl")

    # Skip if already processed
    if os.path.exists(embedding_file):
        print(f"✅ Already processed: {file_name}")
        continue

    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    # if len(text) < 300:
    #     continue

    # chunks = chunk_text(text)
    chunks = text_splitter.split_text(text)
    total_chunks_created += len(chunks)

    # --- Time only the embedding generation step ---
    start_time = time.time()
    embeddings = model.encode(chunks, batch_size=64, show_progress_bar=False)
    elapsed = time.time() - start_time
    total_embedding_time += elapsed

    print(f"⏱️ {file_name}: {len(chunks)} chunks embedded in {elapsed:.2f}s")

    # Save to Drive
    data_to_save = {
        "chunks": chunks,
        "embeddings": embeddings,
        "file_name": file_name,
        "book_title": base_name
    }

    with open(embedding_file, "wb") as f:
        pickle.dump(data_to_save, f)

    print(f"💾 Saved embeddings for {file_name}")

    # Clean memory
    del chunks, embeddings
    torch.cuda.empty_cache()
    gc.collect()

# --- Final summary ---
print("\n" + "="*50)
print(f"📊 Total chunks created: {total_chunks_created}")
print(f"⏱️ Total embedding generation time: {total_embedding_time:.2f} seconds "
      f"({total_embedding_time/60:.2f} minutes)")
if total_chunks_created > 0:
    print(f"⚡ Average time per chunk: {total_embedding_time/total_chunks_created:.4f} seconds")
print("="*50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

custom_st.py: 0.00B [00:00, ?B/s]

...

⏱️ کشکول فاطمی -()_19793.html: 555 chunks embedded in 49.25s
💾 Saved embeddings for کشکول فاطمی -()_19793.html


 99%|█████████▉| 99/100 [1:30:57<00:42, 42.58s/it]

⏱️ گلشن ياس نبي -()_16106.html: 1116 chunks embedded in 117.29s
💾 Saved embeddings for گلشن ياس نبي -()_16106.html


100%|██████████| 100/100 [1:32:56<00:00, 55.77s/it]


📊 Total chunks created: 61934
⏱️ Total embedding generation time: 5486.74 seconds (91.45 minutes)
⚡ Average time per chunk: 0.0886 seconds


# Load Saved Embeddings

In [7]:
import os
import pickle
import numpy as np
from tqdm.notebook import tqdm

save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa5_nano"

all_chunks_text = []
all_embeddings = []
all_metadata = []

pkl_files = [f for f in os.listdir(save_folder) if f.endswith('.pkl')]
print(f"Found {len(pkl_files)} embedding files")

for pkl_file in tqdm(pkl_files, desc="Loading embeddings"):
    file_path = os.path.join(save_folder, pkl_file)
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    chunks     = data["chunks"]
    embeddings = data["embeddings"]
    book_title = data["book_title"]
    file_name  = data["file_name"]

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        all_chunks_text.append(chunk)
        all_embeddings.append(emb)
        all_metadata.append({
            "book_title": book_title,
            "file_name":  file_name,
            "chunk_id":   i
        })

all_embeddings = np.array(all_embeddings).astype('float32')

print(f"""
============== LOADED DATA ==============
  Total books:      {len(pkl_files)}
  Total chunks:     {len(all_chunks_text)}
  Embedding dim:    {all_embeddings.shape[1]}
  Matrix size:      {all_embeddings.nbytes/1024**2:.2f} MB
==========================================
""")

Found 100 embedding files


Loading embeddings:   0%|          | 0/100 [00:00<?, ?it/s]


============== LOADED DATA ==============
  Total books:      100
  Total chunks:     88275
  Embedding dim:    768
  Matrix size:      258.62 MB



# Qdrant

In [4]:
!pip install -q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 13.7 MB/s eta 0:00:00


In [9]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import time

client_qdrant = QdrantClient(":memory:")   # or QdrantClient(path="/content/qdrant_db") to persist

collection_name = "books_jina5"
dim = all_embeddings.shape[1]

client_qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Uploading to Qdrant...")
upload_start = time.time()

upload_batch = 512
for i in tqdm(range(0, len(all_embeddings), upload_batch), desc="Uploading"):
    batch_end = min(i + upload_batch, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={
                "book_title": all_metadata[j]["book_title"],
                "file_name":  all_metadata[j]["file_name"],
                "chunk_id":   all_metadata[j]["chunk_id"],
                "text":       all_chunks_text[j]
            }
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=collection_name, points=points)

upload_elapsed = time.time() - upload_start

print(f"""
============== QDRANT LOADED ==============
  Collection:    {collection_name}
  Total points:  {client_qdrant.get_collection(collection_name).points_count}
  Upload time:   {upload_elapsed:.2f}s
============================================
""")

Uploading to Qdrant...


Uploading:   0%|          | 0/173 [00:00<?, ?it/s]

/tmp/ipykernel_1000/168165666.py:34: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client_qdrant.upsert(collection_name=collection_name, points=points)



============== QDRANT LOADED ==============
  Collection:    books_jina5
  Total points:  88275
  Upload time:   74.26s



# Build Qdrant index + measure time

In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

client_qdrant = QdrantClient(":memory:")
qdrant_collection = "books_jina5"
dim = all_embeddings.shape[1]

client_qdrant.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Indexing into Qdrant...")
qdrant_index_start = time.time()

upload_batch = 512
for i in tqdm(range(0, len(all_embeddings), upload_batch), desc="Qdrant upload"):
    batch_end = min(i + upload_batch, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={"file_name": all_metadata[j]["file_name"]}
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=qdrant_collection, points=points)

qdrant_index_time = time.time() - qdrant_index_start
print(f"Qdrant indexing time: {qdrant_index_time:.2f}s\n")

Indexing into Qdrant...


Qdrant upload:   0%|          | 0/173 [00:00<?, ?it/s]

Qdrant indexing time: 68.72s



# Weaviate

In [2]:
!pip install -q weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.9 MB/s eta 0:00:00


In [10]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

client_weaviate = weaviate.connect_to_embedded()

collection_name = "BooksJina5"

if client_weaviate.collections.exists(collection_name):
    client_weaviate.collections.delete(collection_name)

client_weaviate.collections.create(
    name=collection_name,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="book_title", data_type=DataType.TEXT),
        Property(name="file_name",  data_type=DataType.TEXT),
        Property(name="chunk_id",   data_type=DataType.INT),
        Property(name="text",       data_type=DataType.TEXT),
    ]
)

print("Uploading to Weaviate...")
upload_start = time.time()

col = client_weaviate.collections.get(collection_name)

with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Uploading"):
        batch.add_object(
            properties={
                "book_title": all_metadata[i]["book_title"],
                "file_name":  all_metadata[i]["file_name"],
                "chunk_id":   all_metadata[i]["chunk_id"],
                "text":       all_chunks_text[i]
            },
            vector=all_embeddings[i].tolist()
        )

upload_elapsed = time.time() - upload_start

print(f"""
============== WEAVIATE LOADED ==============
  Collection:    {collection_name}
  Total objects: {col.aggregate.over_all(total_count=True).total_count}
  Upload time:   {upload_elapsed:.2f}s
==============================================
""")

INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.30.5/weaviate-v1.30.5-Linux-amd64.tar.gz
INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 6616


Uploading to Weaviate...


Uploading:   0%|          | 0/88275 [00:00<?, ?it/s]


============== WEAVIATE LOADED ==============
  Collection:    BooksJina5
  Total objects: 88275
  Upload time:   182.53s



# Build Weaviate Index



In [13]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

client_weaviate = weaviate.connect_to_embedded()
weaviate_collection = "BooksJina5"

if client_weaviate.collections.exists(weaviate_collection):
    client_weaviate.collections.delete(weaviate_collection)

client_weaviate.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="file_name", data_type=DataType.TEXT),
    ]
)

print("Indexing into Weaviate...")
weaviate_index_start = time.time()

col = client_weaviate.collections.get(weaviate_collection)
with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Weaviate upload"):
        batch.add_object(
            properties={"file_name": all_metadata[i]["file_name"]},
            vector=all_embeddings[i].tolist()
        )

weaviate_index_time = time.time() - weaviate_index_start
print(f"Weaviate indexing time: {weaviate_index_time:.2f}s\n")

INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 9120


Indexing into Weaviate...


Weaviate upload:   0%|          | 0/88275 [00:00<?, ?it/s]

Weaviate indexing time: 144.62s



# load benchmark

In [3]:
import json
import pandas as pd

benchmark_file = '/content/drive/MyDrive/benchmark_100books.json'
# ===================== LOAD BENCHMARK =====================
with open(benchmark_file, 'r', encoding='utf-8') as f:
    benchmark = json.load(f)

benchmark_df = pd.DataFrame(benchmark)
print(f"Loaded {len(benchmark_df)} benchmark queries\n")

Loaded 100 benchmark queries



# Run benchmark on both + measure query latency

In [20]:
from weaviate.classes.query import MetadataQuery
from sentence_transformers import SentenceTransformer
import torch

model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"

# ===================== LOAD MODEL (for query encoding) =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")
model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

def evaluate_vector_store(search_fn, name):
    hits_at_1 = 0
    hits_at_5 = 0
    hits_at_10 = 0
    latencies = []

    for _, row in tqdm(benchmark_df.iterrows(), desc=f"Evaluating {name}", total=len(benchmark_df)):
        query = row["query"]
        expected = row["expected_file"]

        # Encode query
        query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()

        # Search (timed)
        t0 = time.time()
        retrieved_files = search_fn(query_emb)
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        if retrieved_files and retrieved_files[0] == expected:
            hits_at_1 += 1
        if expected in retrieved_files[:5]:
            hits_at_5 += 1
        if expected in retrieved_files[:10]:
            hits_at_10 += 1

    n = len(benchmark_df)
    lat = np.array(latencies)

    return {
        "vector_store": name,
        "hit@1": round(hits_at_1 / n, 4),
        "hit@5": round(hits_at_5 / n, 4),
        "hit@10": round(hits_at_10 / n, 4),
        "latency_mean_ms": round(lat.mean(), 2),
        "latency_p95_ms": round(np.percentile(lat, 95), 2),
        "latency_p99_ms": round(np.percentile(lat, 99), 2),
    }

# ---- Qdrant search function ----
def qdrant_search(query_emb, k=50):
    results = client_qdrant.query_points(
        collection_name=qdrant_collection,
        query=query_emb,
        limit=k,
        with_payload=True
    ).points

    retrieved = []
    seen = set()
    for r in results:
        fname = r.payload["file_name"]
        if fname not in seen:
            seen.add(fname)
            retrieved.append(fname)
        if len(retrieved) == 10:
            break
    return retrieved

# ---- Weaviate search function ----
def weaviate_search(query_emb, k=50):
    results = col.query.near_vector(
        near_vector=query_emb,
        limit=k,
        return_properties=["file_name"]
    )

    retrieved = []
    seen = set()
    for r in results.objects:
        fname = r.properties["file_name"]
        if fname not in seen:
            seen.add(fname)
            retrieved.append(fname)
        if len(retrieved) == 10:
            break
    return retrieved

# ===================== RUN EVALUATION =====================
qdrant_results   = evaluate_vector_store(qdrant_search, "Qdrant")
weaviate_results = evaluate_vector_store(weaviate_search, "Weaviate")

Loading model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.22k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

Model loaded!



Evaluating Qdrant:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Evaluating Weaviate:   0%|          | 0/100 [00:00<?, ?it/s]

# Final comparison

In [21]:
comparison_df = pd.DataFrame([qdrant_results, weaviate_results])

print(f"""
==================== INDEXING TIME ====================
  Qdrant:    {qdrant_index_time:.2f}s
  Weaviate:  {weaviate_index_time:.2f}s
=========================================================
""")

print("==================== BENCHMARK RESULTS ====================")
print(comparison_df.to_string(index=False))

# Save for your report
comparison_df.to_csv("/content/drive/MyDrive/vectorstore_comparison.csv", index=False)
print("\nSaved comparison to vectorstore_comparison.csv")


==================== INDEXING TIME ====================
  Qdrant:    68.72s
  Weaviate:  144.62s

==================== BENCHMARK RESULTS ====================
vector_store  hit@1  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms
      Qdrant   0.04    0.2    0.25           426.98          563.11         1114.87
    Weaviate   0.04    0.2    0.25            11.65           21.01           42.06

Saved comparison to vectorstore_comparison.csv


#  ******************* Test jinaa3 ***********************

In [6]:
!pip install -q \
  sentence-transformers==3.3.1 \
  transformers==4.47.1 \
  tokenizers==0.21.0

!pip install -q qdrant-client
!pip install -q weaviate-client

In [1]:
import os
import pickle
import numpy as np
from tqdm.notebook import tqdm

save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa3"

all_chunks_text = []
all_embeddings = []
all_metadata = []

pkl_files = [f for f in os.listdir(save_folder) if f.endswith('.pkl')]
print(f"Found {len(pkl_files)} embedding files")

for pkl_file in tqdm(pkl_files, desc="Loading embeddings"):
    file_path = os.path.join(save_folder, pkl_file)
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    chunks     = data["chunks"]
    embeddings = data["embeddings"]
    book_title = data["book_title"]
    file_name  = data["file_name"]

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        all_chunks_text.append(chunk)
        all_embeddings.append(emb)
        all_metadata.append({
            "book_title": book_title,
            "file_name":  file_name,
            "chunk_id":   i
        })

all_embeddings = np.array(all_embeddings).astype('float32')

print(f"""
============== LOADED DATA ==============
  Total books:      {len(pkl_files)}
  Total chunks:     {len(all_chunks_text)}
  Embedding dim:    {all_embeddings.shape[1]}
  Matrix size:      {all_embeddings.nbytes/1024**2:.2f} MB
==========================================
""")


Found 100 embedding files


Loading embeddings:   0%|          | 0/100 [00:00<?, ?it/s]


============== LOADED DATA ==============
  Total books:      100
  Total chunks:     88275
  Embedding dim:    1024
  Matrix size:      344.82 MB



# Qdrant Index creation (jinaa3)

In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import time

client_qdrant = QdrantClient(":memory:")
qdrant_collection = "books_jina3"
dim = all_embeddings.shape[1]

client_qdrant.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Indexing into Qdrant...")
qdrant_index_start = time.time()

upload_batch = 512
for i in tqdm(range(0, len(all_embeddings), upload_batch), desc="Qdrant upload"):
    batch_end = min(i + upload_batch, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={"file_name": all_metadata[j]["file_name"]}
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=qdrant_collection, points=points)

qdrant_index_time = time.time() - qdrant_index_start
print(f"Qdrant indexing time: {qdrant_index_time:.2f}s\n")

Indexing into Qdrant...


Qdrant upload:   0%|          | 0/173 [00:00<?, ?it/s]

/tmp/ipykernel_13417/3581478147.py:28: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client_qdrant.upsert(collection_name=qdrant_collection, points=points)


Qdrant indexing time: 107.47s



# weaviate for jinaa3

In [9]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

client_weaviate = weaviate.connect_to_embedded()
weaviate_collection = "BooksJina3"

if client_weaviate.collections.exists(weaviate_collection):
    client_weaviate.collections.delete(weaviate_collection)

client_weaviate.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="file_name", data_type=DataType.TEXT),
    ]
)

print("Indexing into Weaviate...")
weaviate_index_start = time.time()

col = client_weaviate.collections.get(weaviate_collection)
with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Weaviate upload"):
        batch.add_object(
            properties={"file_name": all_metadata[i]["file_name"]},
            vector=all_embeddings[i].tolist()
        )

weaviate_index_time = time.time() - weaviate_index_start
print(f"Weaviate indexing time: {weaviate_index_time:.2f}s\n")

Indexing into Weaviate...


Weaviate upload:   0%|          | 0/88275 [00:00<?, ?it/s]

Weaviate indexing time: 196.16s



# test

In [5]:
import json
import pandas as pd
from weaviate.classes.query import MetadataQuery
from sentence_transformers import SentenceTransformer
import torch


benchmark_file = '/content/drive/MyDrive/benchmark_100books.json'
# ===================== LOAD BENCHMARK =====================
with open(benchmark_file, 'r', encoding='utf-8') as f:
    benchmark = json.load(f)

benchmark_df = pd.DataFrame(benchmark)
print(f"Loaded {len(benchmark_df)} benchmark queries\n")


model_name = "jinaai/jina-embeddings-v3"

# ===================== LOAD MODEL (for query encoding) =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")
model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

def evaluate_vector_store(search_fn, name):
    hits_at_1 = 0
    hits_at_5 = 0
    hits_at_10 = 0
    latencies = []

    for _, row in tqdm(benchmark_df.iterrows(), desc=f"Evaluating {name}", total=len(benchmark_df)):
        query = row["query"]
        expected = row["expected_file"]

        # Encode query
        query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()

        # Search (timed)
        t0 = time.time()
        retrieved_files = search_fn(query_emb)
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        if retrieved_files and retrieved_files[0] == expected:
            hits_at_1 += 1
        if expected in retrieved_files[:5]:
            hits_at_5 += 1
        if expected in retrieved_files[:10]:
            hits_at_10 += 1

    n = len(benchmark_df)
    lat = np.array(latencies)

    return {
        "vector_store": name,
        "hit@1": round(hits_at_1 / n, 4),
        "hit@5": round(hits_at_5 / n, 4),
        "hit@10": round(hits_at_10 / n, 4),
        "latency_mean_ms": round(lat.mean(), 2),
        "latency_p95_ms": round(np.percentile(lat, 95), 2),
        "latency_p99_ms": round(np.percentile(lat, 99), 2),
    }

# ---- Qdrant search function ----
def qdrant_search(query_emb, k=50):
    results = client_qdrant.query_points(
        collection_name=qdrant_collection,
        query=query_emb,
        limit=k,
        with_payload=True
    ).points

    retrieved = []
    seen = set()
    for r in results:
        fname = r.payload["file_name"]
        if fname not in seen:
            seen.add(fname)
            retrieved.append(fname)
        if len(retrieved) == 10:
            break
    return retrieved

# ---- Weaviate search function ----
def weaviate_search(query_emb, k=50):
    results = col.query.near_vector(
        near_vector=query_emb,
        limit=k,
        return_properties=["file_name"]
    )

    retrieved = []
    seen = set()
    for r in results.objects:
        fname = r.properties["file_name"]
        if fname not in seen:
            seen.add(fname)
            retrieved.append(fname)
        if len(retrieved) == 10:
            break
    return retrieved

# ===================== RUN EVALUATION =====================
qdrant_results   = evaluate_vector_store(qdrant_search, "Qdrant")
weaviate_results = evaluate_vector_store(weaviate_search, "Weaviate")

comparison_df = pd.DataFrame([qdrant_results, weaviate_results])

print(f"""
==================== INDEXING TIME ====================
  Qdrant:    {qdrant_index_time:.2f}s
  Weaviate:  {weaviate_index_time:.2f}s
=========================================================
""")

print("==================== BENCHMARK RESULTS ====================")
print(comparison_df.to_string(index=False))

# Save for your report
comparison_df.to_csv("/content/drive/MyDrive/vectorstore_comparison_jina3.csv", index=False)
print("\nSaved comparison to vectorstore_comparison.csv")

Loaded 100 benchmark queries

Loading model on cpu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/192 [00:00<?, ?B/s]

Model loaded!



Evaluating Qdrant:   0%|          | 0/100 [00:00<?, ?it/s]

Evaluating Weaviate:   0%|          | 0/100 [00:00<?, ?it/s]

NameError: name 'weaviate_index_time' is not defined

In [7]:

print("==================== BENCHMARK RESULTS ====================")
print(comparison_df.to_string(index=False))

# Save for your report
comparison_df.to_csv("/content/drive/MyDrive/vectorstore_comparison_jina3.csv", index=False)
print("\nSaved comparison to vectorstore_comparison.csv")

==================== BENCHMARK RESULTS ====================
vector_store  hit@1  hit@5  hit@10  latency_mean_ms  latency_p95_ms  latency_p99_ms
      Qdrant   0.14   0.34    0.40           573.78          651.68         1895.94
    Weaviate   0.15   0.33    0.39            47.42          117.28          210.69

Saved comparison to vectorstore_comparison.csv
